# S11 : Automated Rule Library Construction via **SynKit**

<div class="alert alert-block alert-info">
<b>Welcome to SynEdu.</b><br>
This talktorial concludes the <b>SynEdu</b> series by demonstrating how the concepts developed in
<b>S01–S09</b> come together in a realistic, end-to-end workflow: constructing a <b>reaction rule library</b>
from experimental data. We use the <b>USPTO-50K</b> benchmark and rely primarily on <b>SynKit</b>.
</div>

<div class="alert alert-block alert-success">
<b>What you will gain.</b><br>
You will learn an end-to-end, reproducible pipeline:
<ul>
  <li>Atom-map raw reactions (or load cached mappings).</li>
  <li>Canonicalize mapped reactions to remove map-ID / ordering artifacts.</li>
  <li>Construct <b>ITS graphs</b> and extract <b>reaction centers</b>.</li>
  <li>Cluster reaction centers by <b>graph isomorphism</b> (WL prefilter + exact check).</li>
  <li>Export a compact rule library with representatives and members.</li>
</ul>
</div>

<div class="alert alert-block alert-warning">
<b>Prerequisites.</b><br>
You should be comfortable with typed molecular graphs and morphisms (S01),
subgraph matching and MCS (S02), and atom mapping / ITS ideas (S03–S04).
</div>


## Aim of this talktorial

Automated construction of reaction-rule libraries from data relies on two principles:

1. **Canonicalization** — represent equivalent reactions by a unique, deterministic form
   (stable atom identity, stable ordering, stable graph labels).
2. **Equivalence** — identify when two records express the same transformation using
   **graph invariants** and **typed graph isomorphism**, rather than string similarity.

We start from a dataset of reaction records

$$
\mathcal{D} = \{(\mathrm{id}_i,\; c_i,\; \rho_i)\}_{i=1}^{N},
$$

where

$$
\rho_i
$$

is a reaction string (reaction SMILES) and

$$
c_i
$$

is a class label.

We then build a rule library by transforming each record through the pipeline

$$
\rho \;\longmapsto\; \rho^{\mathrm{map}} \;\longmapsto\; \rho^{\mathrm{canon}}
\;\longmapsto\; T \;\longmapsto\; \mathrm{RC}(T),
$$

with the following objects:

$$
\rho^{\mathrm{map}} \;:\; \text{atom-mapped reaction (explicit atom identity)},
$$

$$
\rho^{\mathrm{canon}} \;:\; \text{canonical mapped reaction (stable under map-ID relabeling)},
$$

$$
T \;:\; \text{ITS graph (encodes pre/post bond labels as one typed graph)},
$$

$$
\mathrm{RC}(T) \;:\; \text{reaction-center graph (the core transformation)}.
$$

Rule **equivalence** is defined by typed graph isomorphism

$$
\mathrm{RC}(T_1) \cong \mathrm{RC}(T_2),
$$

after a fast WL-hash prefilter. Each equivalence class becomes one rule (reaction-center pattern),
storing a representative together with member metadata.

---

## Learning outcomes

After completing this talktorial, you will be able to:

- Load and sanity-check a real reaction dataset (USPTO-50K).
- Produce atom-mapped reactions (or load them from cache) and explain why mapping is needed.
- Canonicalize mapped reactions to stabilize identity across map-ID relabelings.
- Construct ITS graphs, extract reaction centers, and compute graph signatures.
- Cluster reaction centers by WL prefiltering and exact typed graph isomorphism.
- Export a compact rule library suitable for downstream rule application.


## Outline

0. **Setup & data**
1. **Load USPTO-50K reactions**
2. **Atom mapping (RXNMapper)**
3. **Canonicalize mapped reactions**
4. **ITS construction and reaction-center extraction**
5. **Reaction-center clustering (WL prefilter + isomorphism)**
6. **Export a compact rule library**
7. **Discussion**
8. **References**


## 0. Setup & data

In [4]:
from pathlib import Path
from typing import Tuple

import pandas as pd
import networkx as nx

from rdkit import Chem
import importlib.metadata as im


def _ver(pkg: str) -> str:
    try:
        return im.version(pkg)
    except Exception:
        return "not-installed"


print("rdkit:", getattr(Chem, "__version__", "unknown"))
print("networkx:", nx.__version__)
print("synkit:", _ver("synkit"))
print("rxnmapper:", _ver("rxnmapper"))

# --- paths (repo-root relative) ---
CSV_PATH = Path("./data/USPTO_50K.csv")
OUTDIR = Path("./out")
OUTDIR.mkdir(parents=True, exist_ok=True)

assert CSV_PATH.exists(), f"Missing input file: {CSV_PATH.resolve()}"
print("Using CSV:", CSV_PATH.resolve())
print("Output dir:", OUTDIR.resolve())

rdkit: unknown
networkx: 3.6.1
synkit: 1.1.0
rxnmapper: 0.4.2
Using CSV: /home/lolo/Documents/TACsy/SynEco/SynEdu/synedu/S10/data/USPTO_50K.csv
Output dir: /home/lolo/Documents/TACsy/SynEco/SynEdu/synedu/S10/out


## 1. Load local USPTO-50K reactions

The provided CSV contains:

- `id`: document/patent id
- `class`: reaction class label (1..10 in many USPTO-50K variants)
- `reactions`: reaction SMILES, typically `reactants>>products` (unmapped)

We normalize to a table with columns:

- `rxn` : unmapped reaction SMILES
- `id`, `class`


In [5]:
df = pd.read_csv(CSV_PATH)

df = df.rename(columns={"reactions": "rxn"})

if "id" not in df.columns and "ID" in df.columns:
    df = df.rename(columns={"ID": "id"})
if "class" not in df.columns and "Class" in df.columns:
    df = df.rename(columns={"Class": "class"})

df["rxn"] = df["rxn"].astype(str).str.strip()

print("Rows:", len(df))
print("Columns:", list(df.columns))
df.head()

Rows: 50016
Columns: ['id', 'class', 'rxn']


,id,class,rxn
0,US05849732,6,COC(=O)[C@H](CCCCNC(=O)OCc1ccccc1)NC(=O)Nc1cc(...
1,US20120114765A1,2,Nc1cccc2cnccc12.O=C(O)c1cc([N+](=O)[O-])c(Sc2c...
2,US08003648B2,1,CCNCC.Cc1nc(-c2ccc(C=O)cc2)sc1COc1ccc([C@H](CC...
3,US09045475B2,1,CC1(C)CCC(CN2CCN(c3ccc(C(=O)NS(=O)(=O)c4ccc(NC...
4,US08188098B2,2,CCOc1ccc(Oc2ncnc3c2cnn3C2CCNCC2)c(F)c1.O=C(Cl)...


**Basic dataset sanity check**

We ensure:
- reactions contain an arrow (`>>` or `>...>`),
- RDKit can parse at least the product side for a small sample.

(We do not attempt to “fix” dataset issues here; rule libraries inherit dataset quality.)


In [6]:
from rdkit import Chem


def split_rxn(rxn: str) -> Tuple[str, str, str]:
    """Return (reactants, reagents, products) from reaction SMILES."""
    rxn = rxn.strip()
    if rxn.count(">") >= 2:
        a, b, c = rxn.split(">", 2)
        return a, b, c
    if ">>" in rxn:
        a, c = rxn.split(">>", 1)
        return a, "", c
    return rxn, "", ""


bad = 0
ncheck = len(df)
for x in df["rxn"].head(ncheck):
    r, g, p = split_rxn(x)
    if not p:
        bad += 1
        continue
    m = Chem.MolFromSmiles(p.split(".")[0])
    if m is None:
        bad += 1

print("Checked:", ncheck)
print("Bad in sample:", bad)

Checked: 50016
Bad in sample: 0


### Exercise Q1 — Sanity-check reaction strings (splitting + basic counts)

**Task.** Add three columns:

- `n_reactants`: number of dot-separated molecules on the left side,
- `n_products`: number of dot-separated molecules on the right side,
- `ok_split`: whether the reaction can be split into two non-empty sides.

Then report the fraction of valid reactions and show the distribution of `n_reactants` / `n_products`.

<details>
<summary><b>Solution</b></summary>

```python
def _count_side(side: str) -> int:
    side = side.strip()
    if not side:
        return 0
    return len([x for x in side.split(".") if x.strip()])

def _split_ok(rxn: str) -> bool:
    try:
        r, _, p = split_rxn(rxn)
        # print(r)
        return bool(r.strip()) and bool(p.strip())
    except Exception:
        return False


df["ok_split"] = df['rxn'].astype(str).apply(_split_ok)
df["n_reactants"] = df['rxn'].astype(str).apply(lambda s: _count_side(split_rxn(s)[0]) if _split_ok(s) else 0)
df["n_products"]  = df['rxn'].astype(str).apply(lambda s: _count_side(split_rxn(s)[2]) if _split_ok(s) else 0)

print("valid fraction:", df["ok_split"].mean())

display(df[["n_reactants","n_products"]].describe())
```
</details>

---

## 2. Atom mapping with RXNMapper

We start from an unmapped reaction SMILES.

$$
\mathrm{rxn} = R > G > P
$$

RXNMapper converts it into an atom-mapped reaction.

$$
\mathrm{rxn}^{\#} = R^{\#} > G > P^{\#}
$$

Mapped reactions are cached to avoid recomputation and ensure determinism.

> **Tip.** Start with  
> $$
> N = 2000
> $$  
> then scale to  
> $$
> N = \lvert \mathrm{USPTO\text{-}50K} \rvert
> $$


In [7]:
from typing import List
from tqdm.auto import tqdm
from rxnmapper import RXNMapper

# How many reactions to map in this run?
N = len(df)  # or e.g. 2000 for a quick test
N = 2000
BATCH = 1


rxnmapper = RXNMapper()

rxns: List[str] = df["rxn"].astype(str).head(N).tolist()
mapped: List[str] = []

for i in tqdm(range(0, len(rxns), BATCH), desc="RXNMapper"):
    batch = rxns[i : i + BATCH]
    out = rxnmapper.get_attention_guided_atom_maps(batch)
    mapped.extend([rec.get("mapped_rxn", "") for rec in out])

# Store directly in df for consistency
df.loc[df.index[:N], "rxn_mapped_raw"] = mapped

df.head()

/home/lolo/miniforge3/envs/synedu/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/lolo/miniforge3/envs/synedu/lib/python3.11/site-packages/rxnmapper/batched_mapper.py:4: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
RXNMapper: 100%|██████████| 2000/2000 [00:43<00:00, 45.51it/s]


,id,class,rxn,rxn_mapped_raw
0,US05849732,6,COC(=O)[C@H](CCCCNC(=O)OCc1ccccc1)NC(=O)Nc1cc(...,O=C(OCc1ccccc1)[NH:10][CH2:9][CH2:8][CH2:7][CH...
1,US20120114765A1,2,Nc1cccc2cnccc12.O=C(O)c1cc([N+](=O)[O-])c(Sc2c...,[NH2:3][c:4]1[cH:5][cH:6][cH:7][c:8]2[cH:9][n:...
2,US08003648B2,1,CCNCC.Cc1nc(-c2ccc(C=O)cc2)sc1COc1ccc([C@H](CC...,[CH3:1][CH2:2][NH:3][CH2:4][CH3:5].O=[CH:6][c:...
3,US09045475B2,1,CC1(C)CCC(CN2CCN(c3ccc(C(=O)NS(=O)(=O)c4ccc(NC...,[CH3:1][C:2]1([CH3:3])[CH2:4][CH2:5][C:6]([CH2...
4,US08188098B2,2,CCOc1ccc(Oc2ncnc3c2cnn3C2CCNCC2)c(F)c1.O=C(Cl)...,[CH3:1][CH2:2][O:3][c:4]1[cH:5][cH:6][c:7]([O:...


### Exercise Q2 — Cache atom-mapped reactions

**Goal.**  
Avoid recomputing atom mappings by caching results to disk.

**Task.**  
Implement a function `load_or_map_reactions(df, cache_path)` that:

1. Checks whether `cache_path` exists.
2. If it exists, loads atom-mapped reactions from disk.
3. Otherwise:
   - runs **RXNMapper** on `df["rxn"]`,
   - saves the mapped reactions to `cache_path`,
   - returns the mapped list.

Assume:
- reactions are stored in column `rxn`,
- caching uses `json.gz`,
- mapping is done in batches.

---

<details>
<summary><b>Solution</b></summary>

```python
import json
import gzip
from pathlib import Path
from typing import List
from tqdm.auto import tqdm

def load_or_map_reactions(
    df,
    cache_path: Path,
    *,
    rxn_col: str = "rxn",
    batch_size: int = 1,
) -> List[str]:
    """
    Load atom-mapped reactions from cache if available;
    otherwise compute them using RXNMapper and cache the result.
    """
    rxns = df[rxn_col].astype(str).tolist()

    # ---- Load from cache ----
    if cache_path.exists() and cache_path.stat().st_size > 0:
        with gzip.open(cache_path, "rt", encoding="utf-8") as f:
            payload = json.load(f)
        return payload["rxn_mapped"]

    # ---- Compute mapping ----
    from rxnmapper import RXNMapper
    mapper = RXNMapper()

    mapped: List[str] = []
    for i in tqdm(range(0, len(rxns), batch_size), desc="RXNMapper"):
        batch = rxns[i : i + batch_size]
        out = mapper.get_attention_guided_atom_maps(batch)
        mapped.extend([rec.get("mapped_rxn", "") for rec in out])

    # ---- Save cache ----
    payload = {"rxn_mapped": mapped}
    with gzip.open(cache_path, "wt", encoding="utf-8") as f:
        json.dump(payload, f)

    return mapped


# Usage
cache_path = OUTDIR / "uspto50k_mapped.json.gz"
df["rxn_mapped"] = load_or_map_reactions(df.iloc[:,:], cache_path)
df.head()
```

In [11]:
import json
import gzip
from pathlib import Path
from typing import List
from tqdm.auto import tqdm


def load_or_map_reactions(
    df,
    cache_path: Path,
    *,
    rxn_col: str = "rxn",
    batch_size: int = 1,
) -> List[str]:
    """
    Load atom-mapped reactions from cache if available;
    otherwise compute them using RXNMapper and cache the result.
    """
    rxns = df[rxn_col].astype(str).tolist()

    # ---- Load from cache ----
    if cache_path.exists() and cache_path.stat().st_size > 0:
        with gzip.open(cache_path, "rt", encoding="utf-8") as f:
            payload = json.load(f)
        return payload["rxn_mapped"]

    # ---- Compute mapping ----
    from rxnmapper import RXNMapper

    mapper = RXNMapper()

    mapped: List[str] = []
    for i in tqdm(range(0, len(rxns), batch_size), desc="RXNMapper"):
        batch = rxns[i : i + batch_size]
        out = mapper.get_attention_guided_atom_maps(batch)
        mapped.extend([rec.get("mapped_rxn", "") for rec in out])

    # ---- Save cache ----
    payload = {"rxn_mapped": mapped}
    with gzip.open(cache_path, "wt", encoding="utf-8") as f:
        json.dump(payload, f)

    return mapped


# Usage
cache_path = OUTDIR / "uspto50k_mapped.json.gz"
df["rxn_mapped"] = load_or_map_reactions(df.iloc[:, :], cache_path)
df.head()

,id,class,rxn,rxn_mapped_raw,rxn_mapped
0,US05849732,6,COC(=O)[C@H](CCCCNC(=O)OCc1ccccc1)NC(=O)Nc1cc(...,O=C(OCc1ccccc1)[NH:10][CH2:9][CH2:8][CH2:7][CH...,O=C(OCc1ccccc1)[NH:10][CH2:9][CH2:8][CH2:7][CH...
1,US20120114765A1,2,Nc1cccc2cnccc12.O=C(O)c1cc([N+](=O)[O-])c(Sc2c...,[NH2:3][c:4]1[cH:5][cH:6][cH:7][c:8]2[cH:9][n:...,[NH2:3][c:4]1[cH:5][cH:6][cH:7][c:8]2[cH:9][n:...
2,US08003648B2,1,CCNCC.Cc1nc(-c2ccc(C=O)cc2)sc1COc1ccc([C@H](CC...,[CH3:1][CH2:2][NH:3][CH2:4][CH3:5].O=[CH:6][c:...,[CH3:1][CH2:2][NH:3][CH2:4][CH3:5].O=[CH:6][c:...
3,US09045475B2,1,CC1(C)CCC(CN2CCN(c3ccc(C(=O)NS(=O)(=O)c4ccc(NC...,[CH3:1][C:2]1([CH3:3])[CH2:4][CH2:5][C:6]([CH2...,[CH3:1][C:2]1([CH3:3])[CH2:4][CH2:5][C:6]([CH2...
4,US08188098B2,2,CCOc1ccc(Oc2ncnc3c2cnn3C2CCNCC2)c(F)c1.O=C(Cl)...,[CH3:1][CH2:2][O:3][c:4]1[cH:5][cH:6][c:7]([O:...,[CH3:1][CH2:2][O:3][c:4]1[cH:5][cH:6][c:7]([O:...


## 3. Canonicalize mapped reactions

Atom-mapped reactions are not unique due to variable component order
and atom-map numbering.
We therefore enforce a canonical representation.

$$
\mathrm{rxn}^{\#} \;\longmapsto\; \mathrm{canon}(\mathrm{rxn}^{\#})
$$

Canonicalization ensures that chemically equivalent reactions admit
an identical atom-mapped form and can be compared structurally.

We use SynKit’s canonicalization based on
Weisfeiler–Lehman refinement, which yields a fast, deterministic but
approximate canonical form; for exact canonicalization, SynKit also
supports nauty-like algorithms with full isomorphism guarantees.

In [13]:
df

,id,class,rxn,rxn_mapped_raw,rxn_mapped
0,US05849732,6,COC(=O)[C@H](CCCCNC(=O)OCc1ccccc1)NC(=O)Nc1cc(...,O=C(OCc1ccccc1)[NH:10][CH2:9][CH2:8][CH2:7][CH...,O=C(OCc1ccccc1)[NH:10][CH2:9][CH2:8][CH2:7][CH...
1,US20120114765A1,2,Nc1cccc2cnccc12.O=C(O)c1cc([N+](=O)[O-])c(Sc2c...,[NH2:3][c:4]1[cH:5][cH:6][cH:7][c:8]2[cH:9][n:...,[NH2:3][c:4]1[cH:5][cH:6][cH:7][c:8]2[cH:9][n:...
2,US08003648B2,1,CCNCC.Cc1nc(-c2ccc(C=O)cc2)sc1COc1ccc([C@H](CC...,[CH3:1][CH2:2][NH:3][CH2:4][CH3:5].O=[CH:6][c:...,[CH3:1][CH2:2][NH:3][CH2:4][CH3:5].O=[CH:6][c:...
3,US09045475B2,1,CC1(C)CCC(CN2CCN(c3ccc(C(=O)NS(=O)(=O)c4ccc(NC...,[CH3:1][C:2]1([CH3:3])[CH2:4][CH2:5][C:6]([CH2...,[CH3:1][C:2]1([CH3:3])[CH2:4][CH2:5][C:6]([CH2...
4,US08188098B2,2,CCOc1ccc(Oc2ncnc3c2cnn3C2CCNCC2)c(F)c1.O=C(Cl)...,[CH3:1][CH2:2][O:3][c:4]1[cH:5][cH:6][c:7]([O:...,[CH3:1][CH2:2][O:3][c:4]1[cH:5][cH:6][c:7]([O:...
...,...,...,...,...,...
50011,US20140194411A1,9,CCOC(=O)N1CCc2ccc3c(c2CC1)C(O)(C1CC1)CC3>>CCOC...,nan,O[C:17]1([CH:18]2[CH2:19][CH2:20]2)[c:13]2[c:1...
50012,US20090149445A1,9,Brc1cccc(C=C2c3ccccc3CCc3ccccc32)c1.N#C[Cu]>>N...,nan,Br[c:3]1[cH:4][cH:5][cH:6][c:7]([CH:8]=[C:9]2[...
50013,US08710243B2,9,Cc1noc(C)c1-c1c(-c2ccc(O)cc2)c2ccccc2n1C=O.NO>...,nan,O=[CH:24][n:23]1[c:8](-[c:7]2[c:2]([CH3:1])[n:...
50014,US20130303532A1,9,O=C(NC1CC1)c1ccc(-c2cnc3c(NCCCO)nc(Br)cn23)cc1...,nan,Br[c:22]1[n:21][c:15]([NH:16][CH2:17][CH2:18][...


Canonicalize:   4%|▍         | 1987/50016 [00:19<02:33, 312.45it/s]

In [14]:
from tqdm import tqdm
from synkit.Chem.Reaction.canon_rsmi import CanonRSMI


def canon_aam(aam: str):
    canon = CanonRSMI(backend="wl", wl_iterations=3).canonicalise(aam)
    return canon.canonical_rsmi


df["rxn_mapped_canon"] = [
    canon_aam(x) if isinstance(x, str) else ""
    for x in tqdm(df["rxn_mapped"], desc="Canonicalize")
]

df = df[["id", "class", "rxn_mapped_canon"]]
df.head()

Canonicalize: 100%|██████████| 50016/50016 [02:45<00:00, 302.79it/s]


,id,class,rxn_mapped_canon
0,US05849732,6,[cH:1]1[cH:9][cH:21][cH:10][cH:2][c:36]1[CH2:3...
1,US20120114765A1,2,[cH:1]1[cH:14][c:10]2[c:23]([cH:11][n:25]1)[cH...
2,US08003648B2,1,[CH3:2][CH2:45][NH:40][CH2:46][CH3:3].[s:1]1[c...
3,US09045475B2,1,[C:1]1([CH3:58])([CH3:59])[CH2:39][CH2:41][C:5...
4,US08188098B2,2,[CH3:1][CH2:8][O:31][c:16]1[cH:6][cH:11][c:34]...


In [ ]:
tmp = df.copy()

n_mapped = len(tmp)
n_unique_canon = tmp["rxn_mapped_canon"].nunique()
print("n_mapped:", n_mapped)
print("n_unique_canon:", n_unique_canon)
print("compression:", n_unique_canon / max(n_mapped, 1))

n_mapped: 50016
n_unique_canon: 49614
compression: 0.9919625719769674


In [ ]:
# # drop duplicates
# df.drop_duplicates(subset='rxn_mapped_canon', inplace=True)
# df.shape

### Exercise Q3 — Canonicalization and deduplication rate

**Goal.**  
Quantify how canonicalization reduces representational redundancy in
atom-mapped reactions.

**Task.**

1. Compute how many mapped reactions collapse after canonicalization.
2. Deduplicate the dataset using the canonical representation.

Report:

- `n_mapped`: total number of mapped reactions,
- `n_unique_canon`: number of unique canonical mapped reactions,
- `compression = n_unique_canon / n_mapped`.

---

<details>
<summary><b>Solution</b></summary>

```python
# Work on a copy to avoid side effects
tmp = df.copy()

# Deduplication statistics
n_mapped = len(tmp)
n_unique_canon = tmp["rxn_mapped_canon"].nunique()
compression = n_unique_canon / max(n_mapped, 1)

print("n_mapped:", n_mapped)
print("n_unique_canon:", n_unique_canon)
print("compression:", compression)

# Deduplicate by canonical atom-mapped reaction
df = tmp.drop_duplicates(subset="rxn_mapped_canon")
print("Shape after deduplication:", df.shape)
```


## 4. ITS construction and reaction-center extraction

Each canonical atom-mapped reaction is represented as an
**Imaginary Transition State (ITS)** graph, which encodes bond changes
directly without explicitly materializing reactants and products.

$$
\mathrm{canon}(\mathrm{rxn}^{\#}) \;\longmapsto\; \mathrm{ITS}
$$

An atom or bond belongs to the **reaction center** if its label differs
between the reactant and product states.

$$
\mathrm{RC} \;\subseteq\; \mathrm{ITS}
$$

The reaction center therefore captures precisely *what changed* during
the reaction.

We prefer SynKit’s ITS construction, which preserves
typed atom and bond annotations and supports direct extraction of the
reaction center.

---

In [ ]:
from synkit.IO import rsmi_to_its
from tqdm import tqdm


def _to_its(aam):
    return rsmi_to_its(aam)


def _to_rc(aam):
    return rsmi_to_its(aam, core=True)


df["ITS"] = [
    _to_its(x) if isinstance(x, str) else None
    for x in tqdm(df["rxn_mapped_canon"], desc="ITS construction")
]

df["RC"] = [
    _to_rc(x) if isinstance(x, str) else None
    for x in tqdm(df["rxn_mapped_canon"], desc="Reaction centers")
]

### Exercise Q4 — Inspect an ITS and its reaction center for one example

**Task.** Pick one canonical mapped reaction, construct its ITS graph, and print:

- number of ITS nodes / edges,
- number of reaction-center edges (core),
- a small list of edges whose bond labels changed.

<details>
<summary><b>Solution</b></summary>

```python
from synkit.IO import rsmi_to_its

aam = df.loc[0, "rxn_mapped_canon"]
T = rsmi_to_its(aam, core=False)
RC = rsmi_to_its(aam, core=True)

print("ITS nodes:", T.number_of_nodes(), "edges:", T.number_of_edges())
print("RC  nodes:", RC.number_of_nodes(), "edges:", RC.number_of_edges())

changed = []
for u, v, ed in T.edges(data=True):
    # convention: ITS edge attributes include pre/post labels; adjust keys if needed
    pre = ed.get("bond_pre") or ed.get("bR") or ed.get("pre")
    post = ed.get("bond_post") or ed.get("bP") or ed.get("post")
    if pre != post:
        changed.append((u, v, pre, post))

print("changed edges (u,v,pre,post) sample:", changed[:10])
```
</details>

---

In [18]:
from synkit.IO import rsmi_to_its

aam = df.loc[0, "rxn_mapped_canon"]
T = rsmi_to_its(aam, core=False)
RC = rsmi_to_its(aam, core=True)

print("ITS nodes:", T.number_of_nodes(), "edges:", T.number_of_edges())
print("RC  nodes:", RC.number_of_nodes(), "edges:", RC.number_of_edges())

changed = []
for u, v, ed in T.edges(data=True):
    # convention: ITS edge attributes include pre/post labels; adjust keys if needed
    pre = ed.get("bond_pre") or ed.get("bR") or ed.get("pre")
    post = ed.get("bond_post") or ed.get("bP") or ed.get("post")
    if pre != post:
        changed.append((u, v, pre, post))

print("changed edges (u,v,pre,post) sample:", changed[:10])

ITS nodes: 37 edges: 38
RC  nodes: 11 edges: 11
changed edges (u,v,pre,post) sample: []


In [21]:
df.loc[0, "rxn_mapped_canon"]

'[cH:1]1[cH:9][cH:21][cH:10][cH:2][c:36]1[CH2:35][O:15][C:23](=[O:13])[NH:16][CH2:34][CH2:4][CH2:6][CH2:17][CH:26]([NH:7][C:8](=[O:5])[NH:25][c:14]1[c:27]([OH:12])[c:29]([C:3]([CH3:18])([CH3:19])[CH3:20])[cH:32][c:37]([O:28][CH3:33])[cH:30]1)[C:24](=[O:22])[O:31][CH3:11]>>[C:3]([CH3:18])([CH3:19])([CH3:20])[c:29]1[c:27]([OH:12])[c:14]([NH:25][C:8](=[O:5])[NH:7][CH:26]([CH2:17][CH2:6][CH2:4][CH2:34][NH2:16])[C:24](=[O:22])[O:31][CH3:11])[cH:30][c:37]([O:28][CH3:33])[cH:32]1'

In [20]:
from synkit.Graph import print_graph_attributes

print_graph_attributes(RC)

🔹 Nodes and their attributes:
  Node 1: {'element': 'C', 'charge': 0, 'typesGH': (('C', True, 1, 0, ['C', 'C']), ('*', False, 0, 0, ['', ''])), 'atom_map': 1}
  Node 36: {'element': 'C', 'charge': 0, 'typesGH': (('C', True, 0, 0, ['C', 'C', 'C']), ('*', False, 0, 0, ['', ''])), 'atom_map': 36}
  Node 9: {'element': 'C', 'charge': 0, 'typesGH': (('C', True, 1, 0, ['C', 'C']), ('*', False, 0, 0, ['', ''])), 'atom_map': 9}
  Node 21: {'element': 'C', 'charge': 0, 'typesGH': (('C', True, 1, 0, ['C', 'C']), ('*', False, 0, 0, ['', ''])), 'atom_map': 21}
  Node 10: {'element': 'C', 'charge': 0, 'typesGH': (('C', True, 1, 0, ['C', 'C']), ('*', False, 0, 0, ['', ''])), 'atom_map': 10}
  Node 2: {'element': 'C', 'charge': 0, 'typesGH': (('C', True, 1, 0, ['C', 'C']), ('*', False, 0, 0, ['', ''])), 'atom_map': 2}
  Node 35: {'element': 'C', 'charge': 0, 'typesGH': (('C', False, 2, 0, ['C', 'O']), ('*', False, 0, 0, ['', ''])), 'atom_map': 35}
  Node 15: {'element': 'O', 'charge': 0, 'typesGH': (

## 5. Reaction-center clustering (WL prefilter + exact isomorphism)

Reaction centers are grouped into **reaction rules** by testing
**typed graph isomorphism**.
Two reaction centers belong to the same rule if they are structurally
identical up to relabeling.

To scale to thousands of reactions, we use a two-stage strategy.

---

### Clustering strategy

1. **WL hash prefilter**  
   Reaction centers are first grouped by a Weisfeiler–Lehman hash,
   which is invariant under graph isomorphism.

2. **Exact typed isomorphism**  
   Within each WL bucket, exact graph isomorphism is tested using
   typed node and edge matches.

3. **Export clusters**  
   Each cluster corresponds to one reaction rule, represented by
   a canonical reaction-center graph and its member indices.

Formally, clustering identifies equivalence classes under isomorphism.

$$
\mathrm{RC}_i \;\sim\; \mathrm{RC}_j
\quad\Longleftrightarrow\quad
\mathrm{RC}_i \cong \mathrm{RC}_j
$$

---

In [ ]:
from synkit.Graph.Feature.wl_hash import WLHash
from tqdm import tqdm


def _wl_sig(graph):
    return WLHash().weisfeiler_lehman_graph_hash(graph)


df["its_sig"] = [
    _wl_sig(x) if x is not None else None for x in tqdm(df["ITS"], desc="ITS WL hash")
]

df["rc_sig"] = [
    _wl_sig(x) if x is not None else None for x in tqdm(df["RC"], desc="RC WL hash")
]

In [ ]:
from synkit.Graph.Matcher.graph_cluster import GraphCluster

data = df.to_dict("records")
cls = GraphCluster()
data = cls.fit(data, rule_key="RC", attribute_key="rc_sig")
data = pd.DataFrame(data)
data.head()

In [ ]:
data["class"].value_counts()

### Exercise Q5 — Implement a minimal reaction-center clustering (WL prefilter + isomorphism)

**Task.** For a small subset of reaction centers, build clusters where two RC graphs are in the same cluster
iff they are isomorphic under your typing predicates.

Return a list of clusters (each cluster is a list of row indices).

<details>
<summary><b>Solution</b></summary>

```python
from synkit.Graph.Feature.wl_hash import WLHash
from networkx.algorithms import isomorphism as iso

def rc_signature(G):
    return WLHash(graph=G).lehman_graph_hash(graph)

def cluster_rc_graphs(rc_graphs):
    # Step 1: bucket by fast signature
    buckets = {}
    for i, G in enumerate(rc_graphs):
        buckets.setdefault(rc_signature(G), []).append(i)

    # Step 2: within each bucket, do exact isomorphism clustering
    clusters = []
    for _, idxs in buckets.items():
        reps = []  # representative index for each cluster
        for i in idxs:
            placed = False
            for rep in reps:
                GM = iso.GraphMatcher(
                    rc_graphs[i], rc_graphs[rep],
                    node_match=node_match,
                    edge_match=edge_match,
                )
                if GM.is_isomorphic():
                    # append to existing cluster
                    for cl in clusters:
                        if rep in cl:
                            cl.append(i)
                            break
                    placed = True
                    break
            if not placed:
                reps.append(i)
                clusters.append([i])
    return clusters

# Example on a small sample
sample = df["RC"].iloc[:200].tolist()
clusters = cluster_rc_graphs(sample)
print("clusters:", len(clusters))
print("cluster sizes (top 10):", sorted([len(c) for c in clusters], reverse=True)[:10])
```
</details>

## 6. Export a compact rule library

We export:

- `rule_library.pkl.gz`: one record per cluster
  - representative core graph (nodes + edges + labels)
  - member indices
  - optional metadata (class distribution, example ids)

In [ ]:
from synkit.Utils.utils import stratified_random_sample
from synkit.IO import save_to_pickle

# Write JSON.GZ
lib_path = OUTDIR / "rule_library.pkl.gz"

df = data[["id", "class", "RC"]]
rule = stratified_random_sample(
    df.to_dict("records"), property_key="class", samples_per_class=1
)
save_to_pickle(rule, lib_path)

### Exercise Q6 — Export and validate a compact rule library

**Task.**  
From your deduplicated dataframe (after canonicalization and clustering), export a compact
rule library and validate basic invariants:

1. Sample at most n rules per class (stratified).
2. Export to a CSV file with columns: `id`, `class`, `rxn_mapped_canon`, `rule_id`.
3. Verify that `rule_id` is unique in the exported library.

<details>
<summary><b>Solution</b></summary>

```python
# Assumes you already have a dataframe like `df_rules` with one row per rule
# and a stable identifier `rule_id` (e.g., cluster id) plus the canonical mapped reaction.
# If you used `data` for clustered RC graphs, adapt accordingly.

MAX_PER_CLASS = 200

df_rules = df.copy()

# Example: use `cluster_id` if available, otherwise fall back to index
if "cluster_id" not in df_rules.columns:
    df_rules["cluster_id"] = range(len(df_rules))
df_rules["rule_id"] = df_rules["cluster_id"]

# Stratified sampling per reaction class
df_lib = stratified_random_sample(
    df_rules,
    by="class",
    n=MAX_PER_CLASS,
    random_state=0,
)

# Keep a minimal schema
keep = ["id", "class", "rxn_mapped_canon", "rule_id"]
df_lib = df_lib[keep].copy()

# Validate uniqueness
assert df_lib["rule_id"].is_unique, "rule_id must be unique in the exported library"

OUT_PATH = "rule_library_compact.csv"
df_lib.to_csv(OUT_PATH, index=False)

print("exported:", len(df_lib), "rules ->", OUT_PATH)
print("classes:", df_lib["class"].nunique())
```
</details>


## 7. Discussion

- Determinism vs. speed: WL signatures speed up clustering but must be validated by exact isomorphism.
- Atom mapping quality dominates: mapping errors propagate into ITS and rule identity.
- Canonicalization is essential: without it, the same transformation can appear under many map-ID permutations.
- Practical tip: cache mappings and intermediate artifacts (canonical AAM, ITS) to make experiments reproducible.


### Exercise Q7 — Matching semantics: what changes when you change typing?

**Task.**  
In this notebook, rule equivalence is defined by isomorphism of typed graphs (ITS / RC graphs).
Consider the following two modifications and predict their effect on clustering:

1. Remove `formal_charge` from the node labels used in matching.
2. Ignore bond order in the edge labels (treat single/double as identical).

For each case, explain whether you expect the number of clusters to increase, decrease,
or remain similar, and why.

<details>
<summary><b>Solution (expected reasoning)</b></summary>

- Removing `formal_charge` makes the typing **coarser**.
  More nodes become compatible under `node_match`, so more graphs become isomorphic.
  **Expected effect:** the number of clusters tends to **decrease** (more merges).

- Ignoring bond order also makes the typing **coarser**.
  Many transformations that differ only by order changes become indistinguishable under matching.
  **Expected effect:** the number of clusters tends to **decrease**, possibly sharply,
  because distinct reaction centers may collapse to the same unlabeled topology.

In both cases, the clustering becomes less chemically specific: it may improve recall
for noisy data but risks merging chemically distinct rules.
</details>


## 8. References and further reading

The following resources provide additional context and deeper technical details
related to graph-based reaction modeling, rule extraction, and reaction templates.

- **SynKit**  
  A graph-theoretic toolkit for modeling chemical reactions using typed graphs,
  atom mappings, and rule-based transformations.  
  *JCIM (2025).*  
  https://pubs.acs.org/doi/10.1021/acs.jcim.5c02123

- **SynTemp**  
  A framework for extracting, analyzing, and applying reaction templates with
  an emphasis on structural changes and reaction centers.  
  *JCIM (2024).*  
  https://pubs.acs.org/doi/10.1021/acs.jcim.4c01795

**Suggested background reading**

- Ehrig, H. *et al.* **Fundamentals of Algebraic Graph Transformation** —  
  the foundational reference for DPO rewriting.
- Willett, P. **Chemical similarity searching** —  
  background on substructure and MCS concepts in chemistry.
- Schneider, N. *et al.* **RXNMapper** —  
  attention-based atom mapping for chemical reactions.

These references collectively connect the formal graph-rewriting perspective
used in this notebook with practical cheminformatics workflows.
